# PyTorch × Transformers — Quick Reference Notebook

Nine self-contained sheets covering autograd, attention blocks, a full GPT-style model, training loop, losses, KV-cached generation, distributed tricks, debugging, and a shape cheat-sheet.

## Sheet 01 — Tensors & Autograd Essentials

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

torch.manual_seed(0)
torch.set_float32_matmul_precision('high')   # TF32 on Ampere+

x = torch.randn(4, 8, 512, device=device)    # (B, T, D)
print('shape:', x.shape, 'dtype:', x.dtype)

z = torch.zeros(3, 3, dtype=torch.bfloat16)
z = z.to(device, dtype=torch.float32)
print('zeros shape:', z.shape, 'dtype:', z.dtype)

In [ ]:
# Autograd core
x = torch.randn(3, requires_grad=True)
y = (x ** 2).sum()
y.backward()                 # populates x.grad
print('x.grad:', x.grad)

with torch.no_grad():        # inference / weight updates
    x -= 0.1 * x.grad

x.grad.zero_()               # reset before next step
x_det = x.detach()           # same data, no grad tracking
print('detached requires_grad:', x_det.requires_grad)

In [ ]:
# Shape ops
B, T, H, D = 2, 4, 2, 8
x = torch.randn(B, T, H * D)
print('view:      ', x.view(B, T, H, D).shape)
print('reshape:   ', x.reshape(B, T, H, D).shape)
x2 = torch.randn(B, T, H, D)
print('transpose: ', x2.transpose(1, 2).shape)
print('permute:   ', x2.permute(0, 2, 1, 3).shape)
print('unsqueeze: ', x2.unsqueeze(0).shape)
print('squeeze:   ', torch.randn(1, 3, 1).squeeze(-1).shape)
a = torch.randn(B, T, D)
b = torch.randn(B, T, D)
print('cat dim=-1:', torch.cat([a, b], dim=-1).shape)
print('stack dim=0:', torch.stack([a, b], dim=0).shape)

In [ ]:
# nn.Module skeleton
class Block(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.fc  = nn.Linear(d_model, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.fc(x))

m = Block(512).to(device)
m.train()   # enables dropout / batchnorm updates
m.eval()    # disables them
x_in = torch.randn(2, 4, 512, device=device)
print('Block output shape:', m(x_in).shape)

## Sheet 02 — The Transformer Building Blocks

In [ ]:
# RMSNorm — LLaMA-style, cheaper than LayerNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

norm = RMSNorm(16)
x_test = torch.randn(2, 4, 16)
print('RMSNorm output shape:', norm(x_test).shape)

In [ ]:
# SwiGLU feed-forward
class SwiGLU(nn.Module):
    def __init__(self, d_model, mult=4):
        super().__init__()
        hidden = int(2 * d_model * mult / 3)
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(hidden, d_model,  bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

ffn = SwiGLU(16)
print('SwiGLU output shape:', ffn(x_test).shape)

In [ ]:
# RoPE positional embeddings
def rope_cache(seq_len, d_head, base=10000, device='cpu'):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device) / d_head))
    t    = torch.arange(seq_len, device=device)
    freqs = torch.outer(t, inv_freq)           # (T, d_head/2)
    return torch.cat([freqs, freqs], dim=-1)   # (T, d_head)

def apply_rope(x, freqs):
    x1, x2 = x.chunk(2, dim=-1)
    rot = torch.cat([-x2, x1], dim=-1)
    return x * freqs.cos() + rot * freqs.sin()

freqs = rope_cache(seq_len=4, d_head=16)
print('RoPE freqs shape:', freqs.shape)
q = torch.randn(2, 2, 4, 16)   # (B, H, T, d_head)
print('rotated q shape:', apply_rope(q, freqs).shape)

In [ ]:
# Multi-Head Attention using F.scaled_dot_product_attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model,     bias=False)
        self.drop = dropout

    def forward(self, x, attn_mask=None, kv_cache=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                        # each (B,T,D)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)   # (B,H,T,d)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        if kv_cache is not None:
            k = torch.cat([kv_cache['k'], k], dim=2)
            v = torch.cat([kv_cache['v'], v], dim=2)
            kv_cache['k'], kv_cache['v'] = k, v

        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask,
            dropout_p=self.drop if self.training else 0.0,
            is_causal=attn_mask is None,
        )                                        # (B,H,T,d)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(out)

mha = MultiHeadAttention(d_model=16, n_heads=2)
x_in = torch.randn(2, 4, 16)
print('MHA output shape:', mha(x_in).shape)

In [ ]:
# Pre-norm Transformer block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model)

    def forward(self, x, **kw):
        x = x + self.attn(self.norm1(x), **kw)   # residual #1
        x = x + self.ffn(self.norm2(x))           # residual #2
        return x

blk = TransformerBlock(d_model=16, n_heads=2)
print('TransformerBlock output shape:', blk(x_in).shape)

## Sheet 03 — A Full GPT-Style Decoder

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2, n_heads=2, max_seq=256):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks  = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm_f  = RMSNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight   # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)                    # (B,T,D)
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        logits = self.head(x)                    # (B,T,vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1),
                ignore_index=-100,
            )
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

model = GPT(vocab_size=256, d_model=64, n_layers=2, n_heads=2).to(device)
print(f'{model.num_params()/1e3:.1f}K parameters')

idx = torch.randint(0, 256, (2, 8), device=device)
logits, _ = model(idx)
print('logits shape:', logits.shape)

targets = torch.roll(idx, -1, dims=1)
_, loss = model(idx, targets)
print('loss:', loss.item())

## Sheet 04 — The Training Loop

In [ ]:
import math

# Cosine schedule with warmup
def lr_lambda(step, warmup=10, total=200, min_ratio=0.1):
    if step < warmup:
        return step / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    cos = 0.5 * (1 + math.cos(math.pi * progress))
    return min_ratio + (1 - min_ratio) * cos

opt = torch.optim.AdamW(
    model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1
)
scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
print('LR at step 0:', scheduler.get_last_lr())

In [ ]:
# Production-grade training loop with AMP + gradient accumulation + clipping
scaler      = torch.cuda.amp.GradScaler(enabled=device == 'cuda')
accum_steps = 2
grad_clip   = 1.0

model.train()
for step in range(5):
    x_b = torch.randint(0, 256, (4, 8), device=device)
    y_b = torch.roll(x_b, -1, dims=1)

    with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=device=='cuda'):
        _, loss = model(x_b, y_b)
        loss = loss / accum_steps

    scaler.scale(loss).backward()

    if (step + 1) % accum_steps == 0:
        scaler.unscale_(opt)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        scheduler.step()
        print(f'step {step+1}: loss={loss.item()*accum_steps:.4f}  grad_norm={norm:.3f}  lr={scheduler.get_last_lr()[0]:.2e}')

In [ ]:
# Checkpointing — save & restore
import os, tempfile

ckpt_path = os.path.join(tempfile.mkdtemp(), 'ckpt.pt')
torch.save({
    'model':     model.state_dict(),
    'optimizer': opt.state_dict(),
    'scheduler': scheduler.state_dict(),
    'step':      step,
}, ckpt_path)
print('saved to', ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model'])
opt.load_state_dict(ckpt['optimizer'])
print('restored at step', ckpt['step'])

In [ ]:
# Gradient checkpointing (activation recompute)
from torch.utils.checkpoint import checkpoint

class TransformerBlockCkpt(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model)

    def _forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

    def forward(self, x, use_ckpt=True):
        if use_ckpt and self.training:
            return checkpoint(self._forward, x, use_reentrant=False)
        return self._forward(x)

blk_ckpt = TransformerBlockCkpt(16, 2)
blk_ckpt.train()
x_c = torch.randn(2, 4, 16, requires_grad=True)
out = blk_ckpt(x_c)
out.sum().backward()
print('gradient checkpointing output shape:', out.shape)

In [ ]:
# torch.compile (skip on older PyTorch or CPU-only envs)
import torch
if hasattr(torch, 'compile') and device == 'cuda':
    compiled_model = torch.compile(model, mode='max-autotune')
    print('model compiled')
else:
    compiled_model = model
    print('torch.compile skipped (CPU or unavailable)')

## Sheet 05 — Losses, Masking & Metrics

In [ ]:
B, T, V = 2, 8, 256
logits_demo  = torch.randn(B, T, V)
targets_demo = torch.randint(0, V, (B, T))

# Standard cross-entropy
loss_ce = F.cross_entropy(logits_demo.view(-1, V), targets_demo.view(-1))
print('cross-entropy loss:', loss_ce.item())

# Label smoothing
loss_smooth = F.cross_entropy(
    logits_demo.view(-1, V), targets_demo.view(-1),
    ignore_index=-100, label_smoothing=0.1,
)
print('label-smoothed loss:', loss_smooth.item())

# Perplexity
ppl = torch.exp(loss_ce)
print('perplexity:', ppl.item())

In [ ]:
# Causal mask — used when NOT passing is_causal=True to sdpa
def causal_mask(T, device='cpu'):
    m = torch.full((T, T), float('-inf'), device=device)
    return torch.triu(m, diagonal=1)   # 0 below/at diag, -inf above

print('causal mask (T=5):')
print(causal_mask(5))

In [ ]:
# Padding mask for batched sequences
B, T = 2, 6
# True = real token
pad_mask = torch.tensor([[True, True, True, True, False, False],
                          [True, True, True, True, True,  False]])
attn_mask = pad_mask[:, None, None, :]           # (B,1,1,T)
attn_mask = attn_mask.expand(B, 1, T, T).float()
attn_mask = attn_mask.masked_fill(~pad_mask[:, None, None, :], float('-inf'))
print('padding attn_mask shape:', attn_mask.shape)

## Sheet 06 — Inference & Autoregressive Generation

In [ ]:
# Greedy decode
@torch.no_grad()
def greedy_generate(model, idx, max_new_tokens=10):
    model.eval()
    for _ in range(max_new_tokens):
        logits, _ = model(idx)
        next_tok  = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_tok], dim=1)
    return idx

prompt = torch.randint(0, 256, (1, 4), device=device)
out    = greedy_generate(model, prompt, max_new_tokens=5)
print('greedy output shape:', out.shape)
print('generated ids:', out[0].tolist())

In [ ]:
# Full generation loop with temperature, top-k and top-p (nucleus)
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    model.eval()
    for _ in range(max_new_tokens):
        logits, _ = model(idx)
        logits    = logits[:, -1, :] / temperature   # (B, vocab)

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')

        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            probs = F.softmax(sorted_logits, dim=-1)
            cum   = torch.cumsum(probs, dim=-1)
            mask  = cum - probs > top_p
            sorted_logits[mask] = float('-inf')
            logits.scatter_(1, sorted_idx, sorted_logits)

        probs    = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)
        idx      = torch.cat([idx, next_tok], dim=1)
    return idx

out_sampled = generate(model, prompt, max_new_tokens=5, temperature=0.8, top_k=50)
print('sampled output ids:', out_sampled[0].tolist())

out_nucleus = generate(model, prompt, max_new_tokens=5, temperature=1.0, top_p=0.9)
print('nucleus  output ids:', out_nucleus[0].tolist())

In [ ]:
# Static KV cache (pre-allocated, avoids realloc per step)
n_heads, max_seq, d_head = 2, 64, 32
k_cache = torch.zeros(1, n_heads, max_seq, d_head, device=device)
v_cache = torch.zeros_like(k_cache)

def update_cache(k_cache, v_cache, k, v, pos):
    k_cache[:, :, pos:pos + k.size(2)] = k
    v_cache[:, :, pos:pos + v.size(2)] = v
    return k_cache[:, :, :pos + k.size(2)], v_cache[:, :, :pos + v.size(2)]

# simulate inserting one token at position 3
k_new = torch.randn(1, n_heads, 1, d_head)
v_new = torch.randn(1, n_heads, 1, d_head)
k_out, v_out = update_cache(k_cache, v_cache, k_new, v_new, pos=3)
print('k_cache after insert shape:', k_out.shape)   # (1, H, 4, d_head)

## Sheet 07 — Distributed Training & Memory Tricks

In [ ]:
# Memory audit helpers
if device == 'cuda':
    print('allocated:', torch.cuda.memory_allocated() / 1e9, 'GB')
    print('reserved: ', torch.cuda.memory_reserved()  / 1e9, 'GB')
    print('peak:     ', torch.cuda.max_memory_allocated() / 1e9, 'GB')
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
else:
    print('CUDA not available — memory stats not applicable on CPU')

In [ ]:
# Memory levers ranked by ease (informational)
levers = [
    '1. Mixed precision (bf16/fp16 autocast)',
    '2. Gradient accumulation (smaller micro-batch)',
    '3. Gradient checkpointing (recompute activations)',
    '4. 8-bit optimizer states (bitsandbytes)',
    '5. FSDP / ZeRO sharding across GPUs',
]
for lv in levers:
    print(lv)

In [ ]:
# DDP setup snippet (illustrative — requires multi-GPU torchrun to actually run)
import os

ddp_snippet = '''
# torchrun --nproc_per_node=8 train.py
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group('nccl')
local_rank = int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(local_rank)

model = GPT(...).to(local_rank)
model = DDP(model, device_ids=[local_rank])
'''
print(ddp_snippet)

In [ ]:
# FSDP setup snippet (illustrative)
fsdp_snippet = '''
import functools
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.distributed.fsdp import MixedPrecision

wrap_policy = functools.partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={TransformerBlock},
)
model = FSDP(model, auto_wrap_policy=wrap_policy,
             mixed_precision=MixedPrecision(param_dtype=torch.bfloat16))
'''
print(fsdp_snippet)

## Sheet 08 — Debugging & Profiling

In [ ]:
# Catch NaN / inf as they happen
# torch.autograd.set_detect_anomaly(True)   # slow — debug only

model.train()
x_b = torch.randint(0, 256, (2, 8), device=device)
y_b = torch.roll(x_b, -1, dims=1)
_, loss = model(x_b, y_b)
loss.backward()

if not torch.isfinite(loss):
    print('bad loss!')
    for n, p in model.named_parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            print('  nan/inf grad in:', n)
else:
    print('loss is finite:', loss.item())

In [ ]:
# Gradient norm monitoring
model.train()
opt.zero_grad(set_to_none=True)
_, loss = model(x_b, y_b)
loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f'grad_norm={total_norm.item():.4f}  loss={loss.item():.4f}')
# healthy: stays roughly flat; a spike often precedes a NaN a few steps later

In [ ]:
# The profiler
from torch.profiler import profile, ProfilerActivity

model.eval()
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
) as prof:
    for _ in range(3):
        with torch.no_grad():
            logits, _ = model(x_b)

print(prof.key_averages().table(sort_by='cpu_time_total', row_limit=8))
# prof.export_chrome_trace('trace.json')   # open in chrome://tracing

In [ ]:
# Common failure → likely cause table
failures = [
    ('loss → NaN',   'LR too high, fp16 overflow, bad init'),
    ('loss flat',    'LR too low, frozen params, wrong loss mask'),
    ('CUDA OOM',     'batch/seq too large, no grad checkpointing'),
    ('slow GPU util','CPU-bound dataloader — raise num_workers'),
]
print(f'{"SYMPTOM":<20}  LIKELY CAUSE')
print('-' * 55)
for symptom, cause in failures:
    print(f'{symptom:<20}  {cause}')

## Sheet 09 — Tensor Shape Cheat Sheet

In [ ]:
# Shape trail through one forward pass
B, T, D, H, vocab = 2, 8, 64, 2, 256
d_head = D // H

shapes = {
    'token ids':             (B, T),
    'after embedding':       (B, T, D),
    'QKV projection':        (B, T, 3 * D),
    'split into heads':      (B, H, T, d_head),
    'attention scores':      (B, H, T, T),
    'attention output':      (B, H, T, d_head),
    'FFN hidden':            (B, T, int(2 * D * 4 / 3)),
    'logits':                (B, T, vocab),
    'KV cache (per layer)':  (B, H, T, d_head),
}

print(f'{"STAGE":<26}  SHAPE')
print('-' * 52)
for stage, shape in shapes.items():
    print(f'{stage:<26}  {shape}')

In [ ]:
# Verify shapes by tracing through a real forward pass
model_trace = GPT(vocab_size=256, d_model=64, n_layers=2, n_heads=2).to(device)
idx_trace   = torch.randint(0, 256, (B, T), device=device)

with torch.no_grad():
    tok_emb_out = model_trace.tok_emb(idx_trace)
    print('after embedding:  ', tuple(tok_emb_out.shape))

    blk0 = model_trace.blocks[0]
    qkv_out = blk0.attn.qkv(blk0.norm1(tok_emb_out))
    print('QKV projection:   ', tuple(qkv_out.shape))

    q, k, v = qkv_out.chunk(3, dim=-1)
    q_heads = q.view(B, T, 2, 32).transpose(1, 2)
    print('split into heads: ', tuple(q_heads.shape))

    scores = torch.einsum('bhid,bhjd->bhij', q_heads, q_heads.clone())
    print('attention scores: ', tuple(scores.shape))

    logits_out, _ = model_trace(idx_trace)
    print('logits:           ', tuple(logits_out.shape))

In [ ]:
# Memory & compute estimates
n_layers = 2
param_count_approx = 12 * n_layers * D ** 2
print(f'param count approx (12·L·D²):  {param_count_approx/1e3:.1f}K')
print(f'actual param count:             {model_trace.num_params()/1e3:.1f}K')
print()
print(f'activation memory / layer (train): O(B·T·D) + O(B·H·T²)')
print(f'  = O({B}·{T}·{D}) + O({B}·{H}·{T}²)')
print(f'  = {B*T*D} + {B*H*T*T} floats')
print()
kv_bytes = 2 * B * T * n_layers * D * 4   # float32
print(f'KV-cache memory (float32): 2·B·T·L·D·4 = {kv_bytes} bytes')